# FC Mold G5 - Pipeline Runner

This notebook imports reusable modules from `src/` and runs the analysis pipeline.

**What it does:**
1. Adds `src/` to Python path
2. Imports config, data loading, sequence analysis, pipeline
3. Runs single-strand or multi-strand analysis
4. Outputs `all_results` dict with DataFrames and sequence groups

**Prerequisites:** Cluster with `scipy` installed (standard on ML Runtime).

In [0]:
import sys
import os

# Add the project root to Python path so `from src.xxx import ...` works
project_root = "/Workspace/Users/dieudonne.nkulikiyimfura@se.abb.com/TATAIjmulden_FCMoldG5"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"sys.path includes src: {project_root in sys.path}")

In [0]:
from src.config import CONFIG, STRAND_CONFIGS, METADATA_PATH
from src.data_loading import StrandDataLoader
from src.sequence_analysis import SequenceAnalyzer
from src.pipeline import StrandAnalysisPipeline

print(f"Config: {CONFIG}")
print(f"Strands: {list(STRAND_CONFIGS.keys())}")
print(f"Metadata: {METADATA_PATH}")

In [0]:
# Choose strand: "23_5" or "23_6"
STRAND_TO_ANALYZE = "23_6"

print(f"\n{'='*70}")
print(f"EXECUTING ANALYSIS FOR: {STRAND_TO_ANALYZE}")
print(f"{'='*70}\n")

pipeline = StrandAnalysisPipeline(
    strand_config=STRAND_CONFIGS[STRAND_TO_ANALYZE],
    config=CONFIG,
    spark=spark,
    dbutils=dbutils,
)

results = pipeline.run()

if results["success"]:
    df_seq = results["df_seq"]
    df_raw = results["df_raw"]
    normal_groups = results["normal_groups"]
    print(f"\nResults: {len(df_seq)} sequences, {df_raw.shape[0]:,} raw rows")
    print(f"Disturbance types:\n{df_seq['disturbance_type'].value_counts().to_string()}")
else:
    print(f"Pipeline failed: {results['error']}")

In [0]:
# Uncomment below to run BOTH strands

# all_results = {}
# for strand_id, strand_config in STRAND_CONFIGS.items():
#     print(f"\n{'#'*70}")
#     print(f"# Processing {strand_config.strand_name}")
#     print(f"{'#'*70}")
#     pipeline = StrandAnalysisPipeline(strand_config, CONFIG, spark, dbutils)
#     all_results[strand_id] = pipeline.run()
#
# # Summary
# for sid, res in all_results.items():
#     status = "OK" if res["success"] else "FAILED"
#     n_seq = len(res.get("df_seq", [])) if res["success"] else 0
#     print(f"  {sid}: {status} ({n_seq} sequences)")

In [0]:
# Preview sequence statistics
if results["success"]:
    display(df_seq.head(10))